# Answer Preparation — Top-100 Retrieval per Query

**Where we are:** Between Step 4 (dense retrieval) and Step 5 (KG-enhanced reranking)  
**What we do:** For each query, retrieve the top-100 most similar passages across all FAISS shards  
**How it connects:** These top-100 passages per query will be the input for KG-based reranking (Step 5)

### Pipeline

1. Encode all 31,372 queries with Contriever → save `query_embeddings.npy`
2. For each shard (0–8): load FAISS index on GPU → batch search all queries → save per-shard top-100
3. Merge per-shard results → keep global top-100 per query → save `top100_merged.parquet`

### Output structure

```
data/NQ_answer/
├── query_embeddings.npy          ← (31372, 768) float32, encoded once
├── shard_00/
│   └── top100_shard.parquet      ← query_id, passage_id, score, rank
├── shard_01/
│   └── top100_shard.parquet
...
└── top100_merged.parquet         ← global top-100 after cross-shard merge
```

## 1. Setup

In [ ]:
import numpy as np
import os
import sys
import json
import math
import time
import torch
import polars as pl
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# faiss-gpu: add DLL paths and conda site-packages
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/bin')
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/lib')
sys.path.insert(0, 'C:/Users/Utente/miniconda3/Lib/site-packages')

import faiss

print(f"faiss {faiss.__version__} — GPU devices: {faiss.get_num_gpus()}")
print(f"torch {torch.__version__} — CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# --- Paths ---
QUERIES_PATH = Path("data/NQ_question/qa_all_entities.jsonl")
FAISS_DIR    = Path("data/faiss_index")      # where shard_XX.faiss / shard_XX_ids.npy live
OUTPUT_DIR   = Path("data/NQ_answer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Model (same as embedding.ipynb) ---
MODEL_NAME = "facebook/contriever"
MAX_LENGTH = 512

# --- Retrieval ---
TOP_K      = 100   # passages to retrieve per query per shard
N_SHARDS   = 9     # shard_00 .. shard_08
BATCH_SIZE = 256   # queries per encoding batch (queries are shorter than passages)

# --- Device ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 3. Load Queries

Load all 31,372 queries from `qa_all_entities.jsonl`.  
Each query has entities in both question and answers — these will be used later for KG reranking.  
We use the line index (0-based) as `query_id`.

In [ ]:
# load all queries from the JSONL file
queries = []
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        queries.append(json.loads(line))

query_texts = [q["question"] for q in queries]
n_queries = len(query_texts)

print(f"Loaded {n_queries:,} queries")
print(f"Example: '{query_texts[0]}'")

## 4. Encode Queries with Contriever

Same model and mean-pooling as passage encoding (`embedding.ipynb`).  
Queries are encoded **once** and reused for all shard searches.

In [ ]:
# load Contriever model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}")

In [ ]:
def mean_pooling(model_output, attention_mask):
    """Contriever-style mean pooling: average non-padding tokens."""
    last_hidden = model_output.last_hidden_state
    last_hidden = last_hidden.masked_fill(
        ~attention_mask.unsqueeze(-1).bool(), 0.0
    )
    emb = last_hidden.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)
    return emb


@torch.no_grad()
def encode_batch(texts, model, tokenizer, device, max_length=MAX_LENGTH):
    """Encode a list of strings into float32 numpy embeddings."""
    inputs = tokenizer(
        texts,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)
    output = model(**inputs)
    emb = mean_pooling(output, inputs["attention_mask"])
    return emb.cpu().numpy()

In [ ]:
query_emb_path = OUTPUT_DIR / "query_embeddings.npy"

if query_emb_path.exists():
    # resume support: reuse previously encoded queries
    print("Loading cached query embeddings ...")
    query_embeddings = np.load(query_emb_path)
    print(f"Loaded {query_embeddings.shape}")
else:
    print(f"Encoding {n_queries:,} queries in batches of {BATCH_SIZE} ...")
    t0 = time.time()
    emb_list = []

    for start in tqdm(range(0, n_queries, BATCH_SIZE), desc="Encoding queries"):
        batch = query_texts[start : start + BATCH_SIZE]
        emb_list.append(encode_batch(batch, model, tokenizer, DEVICE))

    # concatenate all batches → (n_queries, 768)
    query_embeddings = np.concatenate(emb_list, axis=0)
    del emb_list

    np.save(query_emb_path, query_embeddings)
    elapsed = time.time() - t0
    print(f"Saved {query_emb_path} — shape {query_embeddings.shape} in {elapsed:.1f}s")

# sanity check
assert query_embeddings.shape == (n_queries, 768), \
    f"Shape mismatch: {query_embeddings.shape} vs expected ({n_queries}, 768)"

## 5. Rebuild Missing FAISS Indices

Some shards may have `.npy` embeddings but no `.faiss` index (e.g., shard_00).  
We rebuild any missing indices before the search loop.

In [ ]:
for shard_idx in range(N_SHARDS):
    shard_npy  = FAISS_DIR / f"shard_{shard_idx:02d}.npy"
    index_path = FAISS_DIR / f"shard_{shard_idx:02d}.faiss"

    # skip if index already exists or embeddings not available
    if index_path.exists() or not shard_npy.exists():
        continue

    print(f"Rebuilding FAISS index for shard {shard_idx:02d} ...", end=" ")
    embeddings = np.load(shard_npy)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, str(index_path))
    print(f"done — {index.ntotal:,} vectors")
    del embeddings, index

print("All available indices ready.")

## 6. Per-Shard Retrieval

For each shard:
1. Load the `.faiss` index → move to GPU
2. Batch search: all 31,372 queries × top-100 in one call
3. Map FAISS positions → passage IDs via `shard_XX_ids.npy`
4. Save as `shard_XX/top100_shard.parquet`

Shards missing from disk are silently skipped.

In [ ]:
for shard_idx in range(N_SHARDS):
    index_path = FAISS_DIR / f"shard_{shard_idx:02d}.faiss"
    ids_path   = FAISS_DIR / f"shard_{shard_idx:02d}_ids.npy"

    # --- Skip missing shards silently ---
    if not index_path.exists() or not ids_path.exists():
        print(f"Shard {shard_idx:02d}: not on disk, skipping")
        continue

    # --- Resume support: skip already-processed shards ---
    shard_out_dir = OUTPUT_DIR / f"shard_{shard_idx:02d}"
    shard_out_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = shard_out_dir / "top100_shard.parquet"

    if parquet_path.exists():
        print(f"Shard {shard_idx:02d}: already processed, skipping")
        continue

    print(f"\nShard {shard_idx:02d}: loading index ...", end=" ")
    t0 = time.time()

    # load FAISS index and move to GPU for fast search
    cpu_index = faiss.read_index(str(index_path))
    gpu_res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
    del cpu_index

    # load passage ID mapping
    shard_ids = np.load(ids_path)
    print(f"{gpu_index.ntotal:,} vectors on GPU")

    # --- Batch search: all queries at once ---
    # scores: (n_queries, TOP_K) — inner product similarity
    # indices: (n_queries, TOP_K) — positions in the FAISS index
    print(f"Searching {n_queries:,} queries × top-{TOP_K} ...", end=" ")
    scores, indices = gpu_index.search(query_embeddings, TOP_K)
    print(f"done in {time.time() - t0:.1f}s")

    # --- Map FAISS positions → passage IDs ---
    # indices contains positions within this shard (0, 1, 2, ...)
    # shard_ids maps those positions to the original passage IDs from the corpus
    passage_ids = shard_ids[indices]  # (n_queries, TOP_K)

    # --- Build result DataFrame ---
    # flatten the 2D arrays into 1D for Polars
    # query_id: each query repeated TOP_K times
    # rank: 0..99 for each query
    query_id_col = np.repeat(np.arange(n_queries, dtype=np.int32), TOP_K)
    rank_col = np.tile(np.arange(TOP_K, dtype=np.int16), n_queries)

    result_df = pl.DataFrame({
        "query_id":   query_id_col,
        "passage_id": passage_ids.ravel().astype(np.int64),
        "score":      scores.ravel().astype(np.float32),
        "rank":       rank_col,
    })

    result_df.write_parquet(parquet_path)
    print(f"Saved {parquet_path} — {len(result_df):,} rows")

    # free GPU memory before next shard
    del gpu_index, gpu_res, shard_ids, scores, indices, passage_ids, result_df
    torch.cuda.empty_cache()

print("\nPer-shard retrieval complete.")

## 7. Merge Cross-Shard Results

For each query, we collected up to 9 × 100 = 900 candidates (one set per shard).  
Now we merge them and keep only the **global top-100** by score.

The merge uses Polars `group_by` + `sort` + `head` — efficient on ~28M rows.

In [ ]:
merged_path = OUTPUT_DIR / "top100_merged.parquet"

if merged_path.exists():
    print(f"Merged file already exists: {merged_path}")
    merged_df = pl.read_parquet(merged_path)
else:
    # collect all per-shard parquets
    shard_dfs = []
    for shard_idx in range(N_SHARDS):
        parquet_path = OUTPUT_DIR / f"shard_{shard_idx:02d}" / "top100_shard.parquet"
        if parquet_path.exists():
            df = pl.read_parquet(parquet_path)
            # add shard_id column to track provenance
            df = df.with_columns(pl.lit(shard_idx).cast(pl.Int8).alias("shard_id"))
            shard_dfs.append(df)
            print(f"Loaded shard {shard_idx:02d}: {len(df):,} rows")
        else:
            print(f"Shard {shard_idx:02d}: not available, skipping")

    if not shard_dfs:
        raise RuntimeError("No shard results found — run section 6 first")

    print(f"\nMerging {len(shard_dfs)} shards ...")
    t0 = time.time()

    # concatenate all shard results
    all_results = pl.concat(shard_dfs)
    del shard_dfs
    print(f"Total candidates: {len(all_results):,}")

    # for each query: sort by score descending, keep top-100, reassign rank
    merged_df = (
        all_results
        .sort("score", descending=True)
        .group_by("query_id", maintain_order=True)
        .head(TOP_K)
        .with_columns(
            pl.col("score")
              .rank(method="ordinal", descending=True)
              .over("query_id")
              .sub(1)
              .cast(pl.Int16)
              .alias("rank")
        )
    )
    del all_results

    merged_df.write_parquet(merged_path)
    elapsed = time.time() - t0
    print(f"Saved {merged_path} — {len(merged_df):,} rows in {elapsed:.1f}s")

# summary stats
print(f"\nQueries with results: {merged_df['query_id'].n_unique():,}")
print(f"Score range: [{merged_df['score'].min():.4f}, {merged_df['score'].max():.4f}]")
print(f"Shards represented: {sorted(merged_df['shard_id'].unique().to_list())}")
merged_df.head(10)

## 8. Validation

Spot-check: pick a sample query, show its top-5 retrieved passages with text from the corpus.

In [ ]:
# pick a sample query to inspect
sample_qid = 0
sample_question = queries[sample_qid]["question"]
sample_answers = queries[sample_qid]["answers"]

print(f"Query {sample_qid}: '{sample_question}'")
print(f"Gold answers: {sample_answers}\n")

# get top-5 for this query from merged results
top5 = (
    merged_df
    .filter(pl.col("query_id") == sample_qid)
    .sort("rank")
    .head(5)
)

print(f"{'Rank':<6} {'Score':<10} {'Passage ID':<12} {'Shard':<7}")
print("-" * 40)
for row in top5.iter_rows(named=True):
    print(f"{row['rank']:<6} {row['score']:<10.4f} {row['passage_id']:<12} {row['shard_id']:<7}")

In [ ]:
# summary: list all output files
print("\nOutput files:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(OUTPUT_DIR)!s:<40} {size_mb:>10,.1f} MB")